# 02. Baseline-модели

В этом ноутбуке обучаются простые baseline-модели:
- DummyClassifier
- LogisticRegression
- KNN


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.preprocessing import (
    load_raw_data,
    clean_recipes,
    clean_interactions,
    build_recipe_target,
    engineer_features,
    prepare_feature_sets,
    make_splits,
)

from src.modeling import (
    make_dummy,
    make_logreg,
    make_knn,
    run_experiment,
)

## Подготовка данных

In [2]:
recipes, interactions = load_raw_data("../data/raw")

recipes = clean_recipes(recipes)
interactions = clean_interactions(interactions)

data = build_recipe_target(recipes, interactions, min_rating_count=10)
data = engineer_features(data)

X_base, X_full, y, base_feature_cols, full_feature_cols = prepare_feature_sets(data)
splits = make_splits(X_base, X_full, y, random_state=42)

recipes duplicates by full row: 0
recipes duplicates by recipe_id: 0
interactions duplicates by full row: 0
raw thresholds: 4.625 4.857142857142857
rounded for report: 4.6 4.9
final recipe-level dataset shape: (19965, 17)

Распределение классов:
recipe_quality_name
normal    7917
bad       6044
good      6004

Доли классов:
recipe_quality_name
normal    0.3965
bad       0.3027
good      0.3007

Проверка диапазонов mean_rating по классам:
                     count     min     max    mean
recipe_quality_name                               
bad                   6044  2.1333  4.6250  4.4028
good                  6004  4.8571  5.0000  4.9348
normal                7917  4.6259  4.8559  4.7505
Количество столбцов после feature engineering: 52
                                name  recipe_id  minutes  contributor_id  submitted                                                                                                                                                                          

## Baseline-эксперименты на базовых признаках

In [3]:
baseline_results = []

baseline_results.append(
    run_experiment(
        name="Dummy_base",
        family="Dummy",
        model=make_dummy(),
        feature_set="base",
        X_tr=splits["X_train_base"],
        y_tr=splits["y_train"],
        X_ev=splits["X_val_base"],
        y_ev=splits["y_val"],
        params={}
    )
)

baseline_results.append(
    run_experiment(
        name="LogReg_base_C1.0",
        family="LogReg",
        model=make_logreg(C=1.0, use_pca=False),
        feature_set="base",
        X_tr=splits["X_train_base"],
        y_tr=splits["y_train"],
        X_ev=splits["X_val_base"],
        y_ev=splits["y_val"],
        params={"C": 1.0}
    )
)

baseline_results.append(
    run_experiment(
        name="KNN_base_k25_distance",
        family="KNN",
        model=make_knn(n_neighbors=25, weights="distance", use_pca=False),
        feature_set="base",
        X_tr=splits["X_train_base"],
        y_tr=splits["y_train"],
        X_ev=splits["X_val_base"],
        y_ev=splits["y_val"],
        params={"n_neighbors": 25, "weights": "distance"}
    )
)

In [4]:
baseline_df = pd.DataFrame(baseline_results).sort_values(
    ["macro_f1", "balanced_accuracy", "weighted_f1"],
    ascending=False
).reset_index(drop=True)

baseline_df

,experiment,family,feature_set,n_features,params,macro_f1,weighted_f1,balanced_accuracy
0,KNN_base_k25_distance,KNN,base,10,"n_neighbors=25, weights=distance",0.3716,0.3811,0.3746
1,LogReg_base_C1.0,LogReg,base,10,C=1.0,0.3579,0.3540,0.3703
2,Dummy_base,Dummy,base,10,-,0.1893,0.2253,0.3333
